# Docker 第3周：Docker Compose — 多容器编排

> **学习目标**：能用 Compose 编排多服务应用，掌握开发和生产环境配置

---

## 从手动到声明式

前两周你用的都是 `docker run` 命令。一个典型的 Web 应用可能要用：

```bash
docker network create mynet
docker run -d --name db --network mynet -e POSTGRES_PASSWORD=... -v pgdata:/var/lib/postgresql/data postgres:16
docker run -d --name redis --network mynet redis:7
docker run -d --name web --network mynet -p 5000:5000 -e DATABASE_URL=... -e REDIS_URL=... -v ./app:/app my-app
docker run -d --name nginx --network mynet -p 80:80 -v ./nginx.conf:/etc/nginx/nginx.conf nginx
```

每次启动都要敲这么多命令，容易出错，没法复用。

### Compose 的解决方案

把上面的命令翻译成一个 `compose.yml` 文件：

```yaml
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: secret
    volumes:
      - pgdata:/var/lib/postgresql/data

  redis:
    image: redis:7

  web:
    build: .
    ports:
      - "5000:5000"
    environment:
      DATABASE_URL: postgresql://postgres:secret@db:5432/postgres
      REDIS_URL: redis://redis:6379
    volumes:
      - ./app:/app

  nginx:
    image: nginx
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf

volumes:
  pgdata:
```

**一条命令启动全部**：`docker compose up -d`

**一条命令停止全部**：`docker compose down`

Compose 让基础设施变成**代码**——可以提交到 Git、做 Code Review、做版本管理。

---

## compose.yml 核心结构

```yaml
services:      # 定义所有服务（容器）
  web:         # 服务名（也是容器名和 DNS 名）
    image: ... # 或者 build: ...
    ports: ...
    volumes: ...
    environment: ...
    depends_on: ...

networks:      # 自定义网络（可选，默认自动创建）
  ...

volumes:       # 命名卷（可选）
  ...
```

### `image` vs `build`

- `image: redis:7` — 用现成的镜像
- `build: .` — 从当前目录的 Dockerfile 构建
- `build:` 也可以指定路径和文件名：
  ```yaml
  build:
    context: ./backend
    dockerfile: Dockerfile.prod
  ```

### 服务间通信

Compose 自动创建网络，**服务名就是 DNS 名**。Web 服务连数据库，不用写 IP，直接写：

```python
DATABASE_URL = "postgresql://postgres:secret@db:5432/postgres"
#                                           ↑ 这就是 compose 中的服务名
```

---

## 实战：Python Web + Redis 计数器

In [ ]:
# 准备项目目录
! mkdir -p /tmp/compose-demo

# ---- 应用代码 ----
%%writefile /tmp/compose-demo/app.py
import os
import redis
from flask import Flask

app = Flask(__name__)

# Redis 连接：用服务名（docker-compose 会自动解析）
redis_host = os.environ.get("REDIS_HOST", "redis")
r = redis.Redis(host=redis_host, port=6379, decode_responses=True)

@app.route("/")
def home():
    count = r.incr("hits")  # 每次访问 +1
    return f"""
    <h1>访问计数器</h1>
    <p>本页面已被访问 <strong>{count}</strong> 次</p>
    <p><small>数据存储在 Redis 中</small></p>
    """

@app.route("/health")
def health():
    return {"status": "ok"}

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
# ---- requirements.txt ----
%%writefile /tmp/compose-demo/requirements.txt
flask==3.0.0
redis==5.0.0

# ---- Dockerfile ----
%%writefile /tmp/compose-demo/Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
EXPOSE 5000
CMD ["python", "app.py"]

In [ ]:
# ---- compose.yml（核心！）----
%%writefile /tmp/compose-demo/compose.yml
services:
  web:
    build: .
    ports:
      - "5000:5000"
    environment:
      REDIS_HOST: redis
    depends_on:
      - redis

  redis:
    image: redis:7-alpine
    volumes:
      - redis-data:/data

volumes:
  redis-data:

In [ ]:
# 启动所有服务
! docker compose -f /tmp/compose-demo/compose.yml up -d

In [ ]:
# 验证
! curl -s http://localhost:5000/
! curl -s http://localhost:5000/
! curl -s http://localhost:5000/
print("\n访问了 3 次，计数器应该显示 3")

In [ ]:
# 查看服务状态
! docker compose -f /tmp/compose-demo/compose.yml ps

# 查看日志
# ! docker compose -f /tmp/compose-demo/compose.yml logs -f

In [ ]:
# 停止并清理
! docker compose -f /tmp/compose-demo/compose.yml down

# 再次启动，Redis 数据还在（因为有命名卷）
! docker compose -f /tmp/compose-demo/compose.yml up -d
! curl -s http://localhost:5000/ | grep "次"
print("计数从 4 开始——说明数据持久化了！")

# 彻底清理（包括卷）
! docker compose -f /tmp/compose-demo/compose.yml down -v

---

## Compose 常用命令

| 命令 | 作用 |
|------|------|
| `docker compose up -d` | 后台启动所有服务 |
| `docker compose down` | 停止并删除容器、网络 |
| `docker compose down -v` | 同时删除卷（数据没了！） |
| `docker compose ps` | 查看服务状态 |
| `docker compose logs -f [服务名]` | 查看日志（-f 实时跟踪） |
| `docker compose exec <服务名> bash` | 进入某个服务的容器 |
| `docker compose build` | 重新构建镜像 |
| `docker compose restart` | 重启服务 |
| `docker compose up -d --build` | 构建并启动（改代码后常用） |

---

## depends_on：启动顺序

```yaml
services:
  web:
    depends_on:
      - redis
      - db
```

`depends_on` 保证：
- **启动顺序**：先启动 redis 和 db，再启动 web
- **停止顺序**：先停 web，再停 redis 和 db

**⚠️ 重要**：`depends_on` 只等容器启动，不等待服务就绪！Redis 容器启动了不代表 Redis 服务已准备好接受连接。

### 等待服务就绪的方案

方案一：在应用代码里加重试逻辑（推荐）

```python
import time
import redis

def connect_redis(host, retries=10, delay=2):
    for i in range(retries):
        try:
            r = redis.Redis(host=host, port=6379)
            r.ping()
            return r
        except redis.ConnectionError:
            if i < retries - 1:
                print(f"等待 Redis 就绪... ({i+1}/{retries})")
                time.sleep(delay)
    raise Exception("无法连接到 Redis")
```

方案二：Compose v3.9+ 支持 `condition: service_healthy`

```yaml
depends_on:
  redis:
    condition: service_healthy
```

但要求 redis 服务定义了 `HEALTHCHECK`。

---

## 卷与配置管理

### bind mount vs 命名卷

| 类型 | 语法 | 适用场景 |
|------|------|----------|
| bind mount | `./app:/app` | 开发时挂载本地代码，改了立即生效 |
| 命名卷 | `redis-data:/data` | 生产环境持久化，由 Docker 管理 |
| 匿名卷 | 只写容器内路径 | 不推荐，不易管理 |

### 环境变量方式对比

```yaml
# 方式 1：直接写（简单直接，适合少量变量）
environment:
  DEBUG: "true"
  DATABASE_URL: postgresql://...

# 方式 2：从文件加载（适合多变量或敏感信息）
env_file:
  - .env

# 方式 3：使用 shell 环境变量
environment:
  - DATABASE_URL=${DATABASE_URL}
# 启动时：DATABASE_URL=xxx docker compose up
```

### .env 文件

Compose 会自动读取目录下的 `.env` 文件，里面的变量可以在 `compose.yml` 中用 `${VAR}` 引用：

```bash
# .env
POSTGRES_PASSWORD=mysecretpassword
DEBUG=true
```

```yaml
# compose.yml
environment:
  POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}
```

---

## 多环境配置

一个常见需求：开发环境和生产环境配置不同。

### 方案：多 compose 文件

```
compose.yml          # 基础配置
compose.dev.yml      # 开发覆盖（挂载代码、开 debug）
compose.prod.yml     # 生产覆盖（开副本、用 nginx）
```

Docker Compose 支持合并多个文件，后面的覆盖前面的：

```bash
# 开发
docker compose -f compose.yml -f compose.dev.yml up

# 生产
docker compose -f compose.yml -f compose.prod.yml up -d
```

In [ ]:
# 多环境演示
%%writefile /tmp/compose-demo/compose.yml
services:
  web:
    build: .
    environment:
      REDIS_HOST: redis
  redis:
    image: redis:7-alpine

%%writefile /tmp/compose-demo/compose.dev.yml
# 开发环境：挂载代码、开 debug、暴露端口
services:
  web:
    ports:
      - "5000:5000"
    environment:
      DEBUG: "true"
    volumes:
      - ./app.py:/app/app.py

%%writefile /tmp/compose-demo/compose.prod.yml
# 生产环境：多副本、资源限制
services:
  web:
    ports:
      - "80:5000"
    environment:
      DEBUG: "false"
    deploy:
      replicas: 3

# 查看合并后的完整配置
! docker compose -f /tmp/compose-demo/compose.yml -f /tmp/compose-demo/compose.dev.yml config | head -30

---

## 常用的 Compose 模式

### 模式 1：Web + 数据库 + 缓存

```yaml
services:
  web:
    build: .
    ports: ["5000:5000"]
    depends_on: [db, redis]
  db:
    image: postgres:16-alpine
    volumes: ["pgdata:/var/lib/postgresql/data"]
    environment:
      POSTGRES_PASSWORD: ${DB_PASSWORD}
  redis:
    image: redis:7-alpine
```

### 模式 2：Web + Worker + 消息队列

```yaml
services:
  web:
    build: .
    command: python app.py
  worker:
    build: .              # 同一份代码
    command: celery -A tasks worker  # 但不同命令
  redis:
    image: redis:7-alpine
```

### 模式 3：Nginx 反向代理

```yaml
services:
  nginx:
    image: nginx
    ports: ["80:80"]
    volumes:
      - ./nginx.conf:/etc/nginx/conf.d/default.conf
    depends_on: [web]
  web:
    build: .
    # 不需要暴露端口给宿主机，nginx 内部转发即可
```

---

## profile：按需启动服务

有些服务不需要一直跑（如调试工具、批量任务），可以用 `profiles` 控制：

```yaml
services:
  web:
    build: .       # 没有 profiles，始终启动
  debug-tools:
    image: busybox
    profiles: ["debug"]
  migration:
    build: .
    command: python migrate.py
    profiles: ["setup"]
```

```bash
# 正常启动（只启动 web）
docker compose up -d

# 加上调试工具
docker compose --profile debug up -d

# 运行一次性迁移任务
docker compose --profile setup run migration
```

---

## 🎯 第3周总结

| 概念 | 一句话 |
|------|--------|
| **Compose 本质** | 用 YAML 声明应用的所有服务，一条命令全生命周期管理 |
| **服务名 = DNS** | Compose 网络内，用服务名直接通信 |
| **depends_on** | 控制启动顺序，但不保证服务就绪 |
| **多 compose 文件** | 基础 + 环境覆盖，灵活管理多环境 |
| **命名卷** | 持久化数据的推荐方式 |
| **env_file / .env** | 管理环境变量和敏感信息 |

---

## 🧪 综合练习：编排一个多服务应用

用 Compose 编排以下服务：

```
nginx (反向代理, 端口 80)
  ↓
web (Flask API, 2 个实例, 端口 5000)
  ↓ 依赖
  ├── db (PostgreSQL, 持久化数据)
  └── redis (缓存)
```

要求：
1. Web 服务带 HeathCheck
2. 用 `.env` 管理密码
3. 分 dev 和 prod 两个环境配置
4. 一行命令启动全部，一行命令清理全部

> 提示：Nginx 配置文件长这样
> ```nginx
> server {
>     listen 80;
>     location / {
>         proxy_pass http://web:5000;
>     }
> }
> ```

In [ ]:
# 你的练习代码写在这里
pass